In [1]:
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
	
# Set such that PDF fonts export in a manner that they
# are editable in illustrator/affinity
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# set to define axes linewidths
matplotlib.rcParams['axes.linewidth'] = 0.5

# this defines some prefactors so inline figures look nice
%matplotlib inline
%config InlineBackend.figure_format='retina'


matplotlib.rc('font', **font)

from tqdm import tqdm
import pickle
from sparrow import Protein
import protfasta
from tqdm.auto import tqdm

In [2]:
import pandas as pd

In [10]:
from shephard.apis import uniprot
from shephard.interfaces import si_domains, si_protein_attributes

from finches import Mpipi_frontend, CALVADOS_frontend
mf = Mpipi_frontend()
cf = CALVADOS_frontend()

/home/wenyuantong/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/finches/forcefields/calvados.py:236: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.038286503882254706' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  r.loc['H','q'] = 1. / ( 1 + 10**(self.pH-6) )


In [5]:
# Replace 'your_file.csv' with the actual path to your CSV file
csv_file_path = "/home/wenyuantong/Desktop/data/RIN4_domain.tsv"

try:
    # Read the CSV file into a DataFrame
    dom1_df = pd.read_csv(csv_file_path, sep="\t")

    print(f"Successfully loaded '{csv_file_path}' into a DataFrame.")
    # Display the first 5 rows of the DataFrame
    display(dom1_df.head())

except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found.")
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")

Successfully loaded '/home/wenyuantong/Desktop/data/RIN4_domain.tsv' into a DataFrame.


,0,1,2,3
0,AvrRpm1a,1,228,N-terminal
1,AvrRpm1h,1,228,N-terminal
2,AvrRpm1d,1,100,N-terminal
3,AvrRpm1e,1,55,N-terminal
4,AvrRpm1f,1,45,N-terminal


In [8]:
#display(dom1_df.tail(10))

In [11]:
HP = uniprot.uniprot_fasta_to_proteome('/home/wenyuantong/Desktop/data/combined.fasta')
#27977

In [12]:
si_domains.add_domains_from_file(HP, '/home/wenyuantong/Desktop/data/RIN4_domain_all_cut.tsv', skip_bad=True, safe=False)

In [13]:
# build set of domains to work with - default in paper is IDRs between 100 and 150 inclusive
LOWER = 30
UPPER = 80
size_d = []
for d in HP.domains:
    #display(d)
    if len(d) >= LOWER and len(d) <= UPPER:
        size_d.append(d)

# construct name-to-sequence mapping
name2seq = {}
for d1 in size_d:
    d1_name = d1.protein.unique_ID + "_" + d1.domain_name
    name2seq[d1_name] = d1.sequence


# construct name-to-sequence mapping
name2domain = {}
for d1 in size_d:
    d1_name = d1.protein.unique_ID + "_" + d1.domain_name
    name2domain[d1_name] = d1

print(f"Found {len(size_d)} IDRs; this will be {len(size_d)**2} epsilon calculations")

Found 820 IDRs; this will be 672400 epsilon calculations


In [14]:
mode = 'mpipi'

In [15]:
# if set to True we recompute from scratch and save the newly computed values
# to the pickle files, if False, we read from recomputed pickle files
RECOMPUTE = True

if mode == 'mpipi':
    if RECOMPUTE:
        cross_interaction = {}
        
        for d1 in tqdm(size_d):
            
            d1_name = d1.protein.unique_ID + "_" + d1.domain_name
            #print(d1_name)
            cross_interaction[d1_name] = {}        
                
            for d2 in size_d:
                d2_name = d2.protein.unique_ID + "_" + d2.domain_name
                
                cross_interaction[d1_name][d2_name] = mf.epsilon(d1.sequence, d2.sequence)
                
            with open('/home/wenyuantong/Desktop/data/15005027/finches/figure_4/cross_interaction_mpipi.pkl', 'wb') as file:
            # Use pickle to serialize the dictionary and write it to the file
                pickle.dump(cross_interaction, file)
    else:
        # load
    
        with open('/home/wenyuantong/Desktop/data/15005027/finches/figure_4/cross_interaction_mpipi.pkl', 'rb') as file:
            # Load the dictionary back from the pickle file
            cross_interaction = pickle.load(file)



  0%|          | 0/820 [00:00<?, ?it/s]

In [16]:
#all_cross_interactions
all_interactions = []
attractive = 0
repulsive = 0
for k1 in cross_interaction:
    tmp = []
    for k2 in cross_interaction:
        tmp.append(cross_interaction[k1][k2])
        if cross_interaction[k1][k2] < 0:
            attractive = attractive+1
            all_interactions.append([k1,k2,(cross_interaction[k1][k2])])
            
        else:
            repulsive = repulsive + 1
            all_interactions.append([k1,k2,(cross_interaction[k1][k2])])
    #all_cross_interactions.append(tmp)
#all_cross_interactions = np.array(all_cross_interactions)

In [17]:
#all_atr_interactions
atr_df = pd.DataFrame(all_interactions, columns=['IDR1', 'IDR2','E_value'])

# Display the DataFrame
display(atr_df)

,IDR1,IDR2,E_value
0,HopX1a_N-terminal_1_1_80,HopX1a_N-terminal_1_1_80,3.790395
1,HopX1a_N-terminal_1_1_80,HopX1a_Middle_1_200_247,2.613458
2,HopX1a_N-terminal_1_1_80,HopX1a_C-terminal_1_298_377,3.844718
3,HopX1a_N-terminal_1_1_80,HopX1c_N-terminal_1_1_80,3.180637
4,HopX1a_N-terminal_1_1_80,HopX1c_Middle_1_212_267,1.629701
...,...,...,...
672395,Q8GYN5_IDP_3_161_211,HopZ5b_C-terminal_1_283_346,0.798845
672396,Q8GYN5_IDP_3_161_211,AvrB1b_N-terminal_1_1_31,1.363066
672397,Q8GYN5_IDP_3_161_211,Q8GYN5_IDP_1_1_80,-0.191302
672398,Q8GYN5_IDP_3_161_211,Q8GYN5_IDP_2_81_160,-0.057814


In [22]:
eff_df = pd.read_csv('/home/wenyuantong/Desktop/data/RIN4_domain_all_cut.tsv',  sep='\t')
display(eff_df.tail())

,0,1,2,3
815,HopZ5b,283,346,C-terminal_1
816,AvrB1b,1,31,N-terminal_1
817,Q8GYN5,1,80,IDP_1
818,Q8GYN5,81,160,IDP_2
819,Q8GYN5,161,211,IDP_3


In [32]:
# Filter df_rin4 to include rows where column 0 is 'Q8GYN5' or starts with 'AvrB1'
df_filtered = eff_df[
    (eff_df['0'] == 'Q8GYN5') | 
    (eff_df['0'].astype(str).str.startswith('AvrB1'))
].copy()

print("Sub-DataFrame containing 'Q8GYN5' and 'AvrB1*' entries:")
display(df_filtered)

Sub-DataFrame containing 'Q8GYN5' and 'AvrB1*' entries:


,0,1,2,3
816,AvrB1b,1,31,N-terminal_1
817,Q8GYN5,1,80,IDP_1
818,Q8GYN5,81,160,IDP_2
819,Q8GYN5,161,211,IDP_3


In [37]:
df_filtered['0'] = df_filtered['0'].str.replace(' ', '')
substrings_to_find = df_filtered['0'].tolist()
display(substrings_to_find[:10])

['AvrB1b', 'Q8GYN5', 'Q8GYN5', 'Q8GYN5']

In [38]:
# Combine the substrings into a single regex pattern using '|' (OR operator)
# This pattern will match if any of the substrings are found.
search_pattern = '|'.join(substrings_to_find)

# Filter the original DataFrame
# The `case=False` argument makes the search case-insensitive
# The `na=False` argument treats NaN values as not containing the substring
sub_df_multi = atr_df[atr_df['IDR1'].str.contains(search_pattern, case=False, na=False)]

#print(f"\nSub-DataFrame containing any of the terms: {substrings_to_find} in 'item_description' column:")
display(sub_df_multi)

,IDR1,IDR2,E_value
669120,AvrB1b_N-terminal_1_1_31,HopX1a_N-terminal_1_1_80,1.506545
669121,AvrB1b_N-terminal_1_1_31,HopX1a_Middle_1_200_247,0.789849
669122,AvrB1b_N-terminal_1_1_31,HopX1a_C-terminal_1_298_377,1.529569
669123,AvrB1b_N-terminal_1_1_31,HopX1c_N-terminal_1_1_80,1.331453
669124,AvrB1b_N-terminal_1_1_31,HopX1c_Middle_1_212_267,0.281861
...,...,...,...
672395,Q8GYN5_IDP_3_161_211,HopZ5b_C-terminal_1_283_346,0.798845
672396,Q8GYN5_IDP_3_161_211,AvrB1b_N-terminal_1_1_31,1.363066
672397,Q8GYN5_IDP_3_161_211,Q8GYN5_IDP_1_1_80,-0.191302
672398,Q8GYN5_IDP_3_161_211,Q8GYN5_IDP_2_81_160,-0.057814


In [39]:
sub_df_not_containing = sub_df_multi[sub_df_multi['IDR2'].str.contains(search_pattern, case=False, na=False)]

#print(f"\nSub-DataFrame containing rows that *do not* contain any of the terms in '{search_pattern}' in 'item_description' column:")
display(sub_df_not_containing)

,IDR1,IDR2,E_value
669936,AvrB1b_N-terminal_1_1_31,AvrB1b_N-terminal_1_1_31,1.655459
669937,AvrB1b_N-terminal_1_1_31,Q8GYN5_IDP_1_1_80,0.531795
669938,AvrB1b_N-terminal_1_1_31,Q8GYN5_IDP_2_81_160,0.710063
669939,AvrB1b_N-terminal_1_1_31,Q8GYN5_IDP_3_161_211,0.828531
670756,Q8GYN5_IDP_1_1_80,AvrB1b_N-terminal_1_1_31,1.372373
670757,Q8GYN5_IDP_1_1_80,Q8GYN5_IDP_1_1_80,-1.181641
670758,Q8GYN5_IDP_1_1_80,Q8GYN5_IDP_2_81_160,-0.785363
670759,Q8GYN5_IDP_1_1_80,Q8GYN5_IDP_3_161_211,-0.300081
671576,Q8GYN5_IDP_2_81_160,AvrB1b_N-terminal_1_1_31,1.832420
671577,Q8GYN5_IDP_2_81_160,Q8GYN5_IDP_1_1_80,-0.785363


In [40]:
output_file_path = '/home/wenyuantong/Desktop/data/RIN4_pos_control_cross.csv'
sub_df_not_containing.to_csv(output_file_path, header=True, index=False)